In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from misc import model_config

In [2]:
main_model_config = (
    model_config.query("main")
    .drop(columns="main")
    .rename(columns={k: f"model_{k}" for k in model_config.columns})
)

new_name = {
    "powermoe": "PowerMoE",
    "llamamoe": "LLaMA-MoE-v1",
    "olmoe": "OLMoE",
    "switch": "SwitchTransformers",
    "llamamoe2": "LLaMA-MoE-v2",
    "jetmoe": "JetMoE",
    "openmoe": "OpenMoE",
    "minicpm": "MiniCPM-MoE",
    "qwen": "Qwen1.5-MoE",
    "deepseek2": "DeepSeek-V2-Lite",
    "deepseek": "DeepSeekMoE",
    "xverse": "XVERSE-MoE",
    "qwen3": "Qwen3",
    "yuan": "Yuan2.0",
    "phi": "Phi-3.5-MoE",
    "grin": "GRIN-MoE",
    "mixtral": "Mixtral-8x7B",
    "jamba": "Jamba-Mini",
    "nllb": "NLLB-MoE",
    "qwen2": "Qwen2",
}

model_colors = {
    key: px.colors.qualitative.Dark24[i] for i, key in enumerate(main_model_config.index.values)
}

seg_lens = (4, 16, 64, 256)
seg_len_colors = {key: px.colors.qualitative.Plotly[i] for i, key in enumerate(seg_lens)}
main_model_config

,model_name,model_abbr,model_type,model_num_params,model_num_layers,model_num_experts,model_top_k,model_attn
key,,,,,,,,
powermoe,PowerMoE-3B,PW,causal,3.30,32,40,8,flash_attention_2
llamamoe,LLaMA-MoE-v1-3.5B,LL1,causal,6.74,32,16,4,eager
olmoe,OLMoE-1B-7B-0125,OL,causal,6.92,16,64,8,flash_attention_2
switch,SwitchTransformers-Base-128,ST,seq2seq,7.42,24,128,1,eager
llamamoe2,LLaMA-MoE-v2-3.8B,LL2,causal,8.03,32,8,2,flash_attention_2
jetmoe,JetMoE-8B,JT,causal,8.52,24,8,2,flash_attention_2
openmoe,OpenMoE-8B,OP,causal,11.86,24,32,2,eager
minicpm,MiniCPM-MoE-8x2B,MC,causal,13.87,40,8,2,flash_attention_2
qwen,Qwen1.5-MoE-A2.7B,QW1,causal,14.32,24,60,4,flash_attention_2


In [3]:
def make_abbr(df):
    return (
        f"{df["model_abbr"]}{"d" if df["is_decoder"] else "e"}"
        if df["model_type"] == "seq2seq"
        else df["model_abbr"]
    )

In [ ]:
root_dir = Path("../output/sch_mpq")

dfs = {
    p.stem: pd.merge(pd.read_parquet(p), main_model_config, left_on="model", right_index=True)
    for p in root_dir.glob("*.parquet")
}

for df in dfs.values():
    df["model"] = df["model"].astype(model_config.index.dtype)

dfs["m"]

,model,is_decoder,dataset,seg_len,cache_m,recall,ci_lb,ci_ub,model_name,model_abbr,model_type,model_num_params,model_num_layers,model_num_experts,model_top_k,model_attn
0,powermoe,True,c4,4,0.003906,0.003906,0.003906,0.003906,PowerMoE-3B,PW,causal,3.30,32,40,8,flash_attention_2
1,powermoe,True,c4,4,0.007812,0.007812,0.007812,0.007812,PowerMoE-3B,PW,causal,3.30,32,40,8,flash_attention_2
2,powermoe,True,c4,4,0.011719,0.011719,0.011719,0.011719,PowerMoE-3B,PW,causal,3.30,32,40,8,flash_attention_2
3,powermoe,True,c4,4,0.015625,0.015625,0.015625,0.015625,PowerMoE-3B,PW,causal,3.30,32,40,8,flash_attention_2
4,powermoe,True,c4,4,0.019531,0.019531,0.019531,0.019531,PowerMoE-3B,PW,causal,3.30,32,40,8,flash_attention_2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
774139,qwen2,True,github,256,7.982143,1.000000,1.000000,1.000000,Qwen2-57B-A14B,QW2,causal,57.41,28,64,8,flash_attention_2
774140,qwen2,True,github,256,7.986607,1.000000,1.000000,1.000000,Qwen2-57B-A14B,QW2,causal,57.41,28,64,8,flash_attention_2
774141,qwen2,True,github,256,7.991071,1.000000,1.000000,1.000000,Qwen2-57B-A14B,QW2,causal,57.41,28,64,8,flash_attention_2
774142,qwen2,True,github,256,7.995536,1.000000,1.000000,1.000000,Qwen2-57B-A14B,QW2,causal,57.41,28,64,8,flash_attention_2


In [5]:
dfs["m"].query("cache_m == 2").groupby(
    ["model", "is_decoder", "seg_len"], as_index=False, observed=True
)[["ci_lb", "ci_ub"]].mean().pivot(index=["model", "is_decoder"], columns="seg_len").swaplevel(
    0, 1, axis=1
).sort_index(
    axis=1
).sort_values(
    (16, "ci_lb"), ascending=False
)

seg_len                    4                   16                  64   \
                         ci_lb     ci_ub     ci_lb     ci_ub     ci_lb   
model     is_decoder                                                     
llamamoe2 True        0.999668  0.999692  0.990005  0.990403  0.981481   
yuan      True        0.949752  0.950817  0.823579  0.825337  0.784863   
powermoe  True        0.925771  0.926737  0.795351  0.796949  0.736707   
qwen3     True        0.930240  0.931756  0.773708  0.776413  0.692158   
phi       True        0.891902  0.893933  0.739149  0.742567  0.666120   
mixtral   True        0.881584  0.882413  0.737461  0.738664  0.659103   
minicpm   True        0.882296  0.883068  0.732048  0.733098  0.649611   
grin      True        0.882464  0.884513  0.727044  0.730466  0.653008   
olmoe     True        0.887168  0.889049  0.720140  0.723371  0.641601   
jetmoe    True        0.857393  0.858152  0.707964  0.708912  0.628621   
llamamoe  True        0.808688  0.809627  0.669381  0.670267  0.600450   
xverse    True        0.777414  0.778738  0.567273  0.569166  0.455339   
jamba     True        0.777835  0.779099  0.566676  0.568432  0.459961   
deepseek2 True        0.778165  0.779617  0.564675  0.566770  0.453709   
qwen2     True        0.755127  0.755910  0.550780  0.551406  0.452936   
deepseek  True        0.769002  0.770455  0.549649  0.551725  0.435246   
qwen      True        0.695683  0.696996  0.455608  0.457372  0.333495   
nllb      True        0.649712  0.651785  0.443600  0.446675  0.371616   
openmoe   True        0.648729  0.650880  0.417407  0.420627  0.317607   
nllb      False       0.619875  0.621277  0.369995  0.372450  0.278475   
switch    False       0.547172  0.548635  0.263228  0.265740  0.167143   
          True        0.551617  0.553310  0.261695  0.264713  0.161339   

seg_len                              256            
                         ci_ub     ci_lb     ci_ub  
model     is_decoder                                
llamamoe2 True        0.982139  0.976722  0.977587  
yuan      True        0.786893  0.767421  0.769828  
powermoe  True        0.738934  0.709436  0.712135  
qwen3     True        0.695913  0.641547  0.646398  
phi       True        0.670746  0.627349  0.632995  
mixtral   True        0.660735  0.617496  0.619520  
minicpm   True        0.650810  0.607039  0.608441  
grin      True        0.657584  0.613523  0.619008  
olmoe     True        0.645890  0.599566  0.604499  
jetmoe    True        0.629832  0.588032  0.589547  
llamamoe  True        0.601500  0.567372  0.568586  
xverse    True        0.457523  0.400691  0.403252  
jamba     True        0.462293  0.405711  0.408655  
deepseek2 True        0.456288  0.396378  0.399451  
qwen2     True        0.453571  0.408957  0.409697  
deepseek  True        0.437764  0.376798  0.379650  
qwen      True        0.335573  0.274010  0.276428  
nllb      True        0.374898  0.313306  0.316840  
openmoe   True        0.321504  0.272167  0.276686  
nllb      False       0.281440  0.233900  0.237009  
switch    False       0.170144  0.128367  0.131630  
          True        0.164386       NaN       NaN

In [6]:
dfs["m"].assign(
    ci_dist=np.maximum(
        dfs["m"]["ci_ub"] - dfs["m"]["recall"], dfs["m"]["recall"] - dfs["m"]["ci_lb"]
    )
).groupby(["model", "is_decoder", "dataset", "seg_len"], as_index=False, observed=True)[
    ["ci_dist"]
].max().pivot(
    index=["model", "is_decoder"], columns=["dataset", "seg_len"], values="ci_dist"
)

dataset                     c4                                  cc2306  \
seg_len                    4         16        64        256       4     
model     is_decoder                                                     
powermoe  True        0.000387  0.000627  0.000856  0.001150  0.000355   
llamamoe  True        0.000379  0.000376  0.000464  0.000592  0.000357   
olmoe     True        0.000826  0.001288  0.001716  0.002064  0.000769   
switch    False       0.000362  0.000982  0.001645  0.002277  0.000522   
          True        0.000492  0.001289  0.001811       NaN  0.000523   
llamamoe2 True        0.000541  0.000713  0.000786  0.000933  0.000505   
jetmoe    True        0.000289  0.000445  0.000579  0.000791  0.000314   
openmoe   True        0.000783  0.001225  0.001698  0.002010  0.000809   
minicpm   True        0.000310  0.000477  0.000608  0.000755  0.000419   
qwen      True        0.000591  0.000848  0.001106  0.001411  0.000631   
deepseek2 True        0.000679  0.001065  0.001379  0.001844  0.000658   
deepseek  True        0.000668  0.001133  0.001381  0.001796  0.000772   
xverse    True        0.000427  0.000602  0.000858  0.001059  0.000364   
qwen3     True        0.000726  0.001108  0.001512  0.002420  0.000728   
yuan      True        0.000480  0.000745  0.000820  0.000941  0.000489   
phi       True        0.000737  0.001159  0.001647  0.002069  0.000803   
grin      True        0.000779  0.001233  0.001654  0.002116  0.000812   
mixtral   True        0.000370  0.000612  0.000759  0.000977  0.000431   
jamba     True        0.000549  0.000783  0.001058  0.001366  0.000595   
nllb      False       0.000443  0.000812  0.001222  0.001504  0.000521   
          True        0.000765  0.001160  0.001332  0.001531  0.000578   
qwen2     True        0.000274  0.000237  0.000260  0.000322  0.000293   

dataset                                                 book            ...  \
seg_len                    16        64        256       4         16   ...   
model     is_decoder                                                    ...   
powermoe  True        0.000637  0.000871  0.001039  0.000454  0.000584  ...   
llamamoe  True        0.000367  0.000421  0.000447  0.000414  0.000281  ...   
olmoe     True        0.001345  0.001553  0.001887  0.000726  0.001247  ...   
switch    False       0.001166  0.001717  0.002184  0.000415  0.001234  ...   
          True        0.001327  0.001732       NaN  0.000524  0.001526  ...   
llamamoe2 True        0.000680  0.000720  0.000801  0.000513  0.000671  ...   
jetmoe    True        0.000398  0.000556  0.000693  0.000301  0.000321  ...   
openmoe   True        0.001271  0.001601  0.001853  0.001073  0.001704  ...   
minicpm   True        0.000697  0.000702  0.000749  0.000303  0.000367  ...   
qwen      True        0.000844  0.001172  0.001455  0.000546  0.000793  ...   
deepseek2 True        0.001092  0.001380  0.001538  0.000570  0.000749  ...   
deepseek  True        0.001123  0.001417  0.001601  0.000556  0.000817  ...   
xverse    True        0.000528  0.000744  0.000865  0.000395  0.000520  ...   
qwen3     True        0.001111  0.001523  0.002059  0.000655  0.001152  ...   
yuan      True        0.000697  0.000822  0.000949  0.000523  0.000825  ...   
phi       True        0.001173  0.001511  0.001982  0.000783  0.001040  ...   
grin      True        0.001223  0.001418  0.001966  0.000772  0.001057  ...   
mixtral   True        0.000635  0.000812  0.000971  0.000433  0.000469  ...   
jamba     True        0.000824  0.000975  0.001278  0.000555  0.000732  ...   
nllb      False       0.000896  0.001108  0.001238  0.000431  0.000890  ...   
          True        0.000974  0.001131  0.001408  0.000590  0.001138  ...   
qwen2     True        0.000254  0.000286  0.000327  0.000345  0.000241  ...   

dataset                  arxiv           stackexchange                      \
seg_len                    64        256           4         16        64    
model     is_deco

In [7]:
mdf = pd.merge(
    dfs["m"]
    .groupby(["model", "is_decoder", "seg_len", "cache_m"], as_index=False, observed=True)[
        ["recall"]
    ]
    .mean(),
    main_model_config,
    left_on="model",
    right_index=True,
)

mdf

,model,is_decoder,seg_len,cache_m,recall,model_name,model_abbr,model_type,model_num_params,model_num_layers,model_num_experts,model_top_k,model_attn
0,powermoe,True,4,0.003906,0.003906,PowerMoE-3B,PW,causal,3.30,32,40,8,flash_attention_2
1,powermoe,True,4,0.007812,0.007812,PowerMoE-3B,PW,causal,3.30,32,40,8,flash_attention_2
2,powermoe,True,4,0.011719,0.011719,PowerMoE-3B,PW,causal,3.30,32,40,8,flash_attention_2
3,powermoe,True,4,0.015625,0.015625,PowerMoE-3B,PW,causal,3.30,32,40,8,flash_attention_2
4,powermoe,True,4,0.019531,0.019531,PowerMoE-3B,PW,causal,3.30,32,40,8,flash_attention_2
...,...,...,...,...,...,...,...,...,...,...,...,...,...
93819,qwen2,True,256,7.982143,1.000000,Qwen2-57B-A14B,QW2,causal,57.41,28,64,8,flash_attention_2
93820,qwen2,True,256,7.986607,1.000000,Qwen2-57B-A14B,QW2,causal,57.41,28,64,8,flash_attention_2
93821,qwen2,True,256,7.991071,1.000000,Qwen2-57B-A14B,QW2,causal,57.41,28,64,8,flash_attention_2
93822,qwen2,True,256,7.995536,1.000000,Qwen2-57B-A14B,QW2,causal,57.41,28,64,8,flash_attention_2


In [ ]:
srp_dir = Path("../output/srp_mpq")

rdf = pd.merge(
    pd.read_parquet(srp_dir / "mg.parquet"), main_model_config, left_on="model", right_index=True
)

sorted_model_keys = rdf.query("is_decoder and seg_len == 16").sort_values(
    "best_f1", ascending=False
)["model"]

cdf = (
    pd.merge(mdf, rdf)
    .query("cache_m.isin((0.5, 1, 1.5, 2, 2.5, 3))")
    .groupby(["seg_len", "cache_m"], as_index=False)
    .apply(lambda df: pd.Series({"corr": df["recall"].corr(df["best_f1"])}), include_groups=False)
)

cdf.pivot(index="seg_len", columns="cache_m", values="corr")

cache_m,0.5,1.0,1.5,2.0,2.5,3.0
seg_len,,,,,,
4,0.965766,0.979819,0.981494,0.957578,0.885824,0.744447
16,0.979510,0.995426,0.997294,0.983222,0.953331,0.918632
64,0.968622,0.990407,0.997676,0.988978,0.964915,0.931327
256,0.955735,0.983635,0.995723,0.990831,0.968026,0.932043


In [ ]:
fig = make_subplots(
    rows=1,
    cols=len(seg_lens),
    shared_xaxes="all",
    horizontal_spacing=0.01,
    subplot_titles=[f"m={seg_len}" for seg_len in seg_lens],
)

font_size = [16, 20, 24, 28]
show_legend = True

for i, seg_len in enumerate(seg_lens):
    col = i + 1

    for j, key in enumerate(main_model_config.index.values):
        tmpdf = mdf.query(f"model == '{key}' and seg_len == {seg_len}")

        if model_config.loc[key, "type"] == "seq2seq":
            for is_decoder in (False, True):
                subdf = tmpdf.query(f"is_decoder == {is_decoder}")

                fig.add_scatter(
                    x=subdf["cache_m"],
                    y=subdf["recall"],
                    hoverinfo="skip",
                    legendgroup=key,
                    line=go.scatter.Line(color=model_colors[key], width=2),
                    mode="lines",
                    name=f"{new_name[key]} ({"De" if is_decoder else "En"}coder)",
                    opacity=1 if is_decoder else 0.5,
                    showlegend=show_legend,
                    row=1,
                    col=col,
                )
        else:
            fig.add_scatter(
                x=tmpdf["cache_m"],
                y=tmpdf["recall"],
                hoverinfo="skip",
                legendgroup=key,
                line=go.scatter.Line(color=model_colors[key], width=2),
                mode="lines",
                name=new_name[key],
                showlegend=show_legend,
                row=1,
                col=col,
            )

        fig.update_xaxes(
            range=[0, 4],
            tickfont=go.layout.xaxis.Tickfont(size=font_size[1]),
            title=go.layout.xaxis.Title(
                font=go.layout.xaxis.title.Font(size=font_size[2]), text="ρ"
            ),
            row=1,
            col=col,
        )

        fig.update_yaxes(range=[0, 1], showticklabels=col == 1, row=1, col=col)

        if col == 1:
            fig.update_yaxes(
                tickfont=go.layout.yaxis.Tickfont(size=font_size[1]),
                title=go.layout.yaxis.Title(
                    font=go.layout.yaxis.title.Font(size=font_size[2]), text="SCH(E,m,ρ)"
                ),
                row=1,
                col=col,
            )

    show_legend = False

fig.update_annotations(font=go.layout.annotation.Font(size=font_size[3]))

fig.update_layout(
    legend=go.layout.Legend(
        font=go.layout.legend.Font(size=font_size[1]),
        itemsizing="constant",
        orientation="h",
        y=-0.15,
        yanchor="top",
    ),
    margin=go.layout.Margin(l=60, r=30, t=30, b=0),
    width=2000,
    height=600,
)

plot_dir = Path("../plot")
plot_dir.mkdir(parents=True, exist_ok=True)
fig.write_image(plot_dir / "msch.pdf")
# fig.show()

In [10]:
ldf = pd.merge(
    dfs["l"]
    .groupby(
        ["model", "is_decoder", "layer_idx", "seg_len", "cache_m"], as_index=False, observed=True
    )[["recall"]]
    .mean(),
    main_model_config,
    left_on="model",
    right_index=True,
)

ldf

,model,is_decoder,layer_idx,seg_len,cache_m,recall,model_name,model_abbr,model_type,model_num_params,model_num_layers,model_num_experts,model_top_k,model_attn
0,powermoe,True,0,4,0.125,0.105307,PowerMoE-3B,PW,causal,3.30,32,40,8,flash_attention_2
1,powermoe,True,0,4,0.250,0.199452,PowerMoE-3B,PW,causal,3.30,32,40,8,flash_attention_2
2,powermoe,True,0,4,0.375,0.285386,PowerMoE-3B,PW,causal,3.30,32,40,8,flash_attention_2
3,powermoe,True,0,4,0.500,0.363603,PowerMoE-3B,PW,causal,3.30,32,40,8,flash_attention_2
4,powermoe,True,0,4,0.625,0.435134,PowerMoE-3B,PW,causal,3.30,32,40,8,flash_attention_2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
93819,qwen2,True,27,256,7.500,0.961658,Qwen2-57B-A14B,QW2,causal,57.41,28,64,8,flash_attention_2
93820,qwen2,True,27,256,7.625,0.972152,Qwen2-57B-A14B,QW2,causal,57.41,28,64,8,flash_attention_2
93821,qwen2,True,27,256,7.750,0.982195,Qwen2-57B-A14B,QW2,causal,57.41,28,64,8,flash_attention_2
93822,qwen2,True,27,256,7.875,0.991622,Qwen2-57B-A14B,QW2,causal,57.41,28,64,8,flash_attention_2


In [ ]:
num_cols = 10
num_rows = (len(main_model_config) - 1) // num_cols + 1

fig = make_subplots(
    rows=num_rows,
    cols=num_cols,
    shared_xaxes="all",
    shared_yaxes="all",
    horizontal_spacing=0.005,
    vertical_spacing=0.11,
    subplot_titles=sorted_model_keys.map(new_name).values,
)

font_size = [16, 16, 18, 20]
show_legend = True

for i, key in enumerate(sorted_model_keys):
    if (ldf["model"] == key).sum() == 0:
        continue

    row = i // num_cols + 1
    col = i % num_cols + 1
    num_layers = model_config.loc[key, "num_layers"]

    for seg_len in seg_lens:
        for j, cache_m in enumerate((1, 2)):
            tmpdf = ldf.query(f"model == '{key}' and seg_len == {seg_len} and cache_m == {cache_m}")

            if len(tmpdf) == 0:
                continue

            layer_idx = tmpdf["layer_idx"] + 1
            if model_config.loc[key, "type"] == "seq2seq":
                layer_idx += tmpdf["is_decoder"] * model_config.loc[key, "num_layers"] // 2

            line_name = f"SCH(E,{seg_len},{cache_m})"

            fig.add_scatter(
                x=layer_idx / num_layers,
                y=tmpdf["recall"],
                hoverinfo="skip",
                legendgroup=line_name,
                line=go.scatter.Line(color=seg_len_colors[seg_len], dash="dot" if j else "solid"),
                marker=go.scatter.Marker(size=4),
                mode="lines" if j else "lines+markers",
                name=line_name,
                opacity=0.5 if j else 1,
                showlegend=show_legend,
                row=row,
                col=col,
            )

    show_legend = False

    fig.update_xaxes(
        showticklabels=True,
        tickfont=go.layout.xaxis.Tickfont(size=font_size[1]),
        tickvals=[1 / num_layers, 0.5, 1],
        ticktext=[1, num_layers // 2, num_layers],
        row=row,
        col=col,
    )

    if row == num_rows:
        fig.update_xaxes(
            title=go.layout.xaxis.Title(
                font=go.layout.xaxis.title.Font(size=font_size[2]), standoff=1, text="Layer"
            ),
            row=row,
            col=col,
        )

    fig.update_yaxes(showticklabels=col == 1, row=row, col=col)

    if col == 1:
        fig.update_yaxes(
            tickfont=go.layout.yaxis.Tickfont(size=font_size[1]),
            title=go.layout.yaxis.Title(
                font=go.layout.yaxis.title.Font(size=font_size[2]), text="SCH(E,m,ρ)"
            ),
            row=row,
            col=col,
        )

fig.update_annotations(font=go.layout.annotation.Font(size=font_size[3]))

fig.update_layout(
    legend=go.layout.Legend(
        font=go.layout.legend.Font(size=font_size[2]), orientation="h", x=0.47, xanchor="center"
    ),
    margin=go.layout.Margin(l=60, r=0, t=30, b=0),
    width=2000,
    height=500,
)

plot_dir = Path("../plot")
plot_dir.mkdir(parents=True, exist_ok=True)
fig.write_image(plot_dir / "lsch.pdf")
# fig.show()